# 1 · Colab setup

Run this notebook **once**, before notebooks 2–6.

### Before running
1. Upload the whole package folder to the **root of your Google Drive** and name it
   **`btc_rl_task`**. It should contain `data/` and notebooks `1_` to `6_`.
2. Open this notebook from that Drive folder in Colab.
3. **Runtime → Change runtime type → GPU (T4 or better)**.
4. **Runtime → Run all.**

### What it does
Restores the compressed dataset, creates the folder layout the training notebooks
expect, and sanity-checks the data. Every notebook works inside `btc_rl_task/`, so
all data, trained models and results persist on Drive across Colab disconnects.

## Runtime check

In [ ]:
import subprocess, psutil

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU        :", gpu or "NONE - enable a GPU under Runtime > Change runtime type")
ram = psutil.virtual_memory().total / 1e9
print(f"System RAM : {ram:.1f} GB")
if ram < 20:
    print("\nStandard-RAM runtime. Alpha101 generation on 1.28M rows peaks at roughly 4-8 GB")
    print("and should fit. If notebook 3 runs out of memory, use the 5-minute fallback below.")

## Mount Drive and restore the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gzip, shutil, time

WORKDIR = "/content/drive/MyDrive/btc_rl_task"
GZ  = f"{WORKDIR}/data/BTC_1min_with_sentiment_risk_train.csv.gz"
CSV = f"{WORKDIR}/data/BTC_1min_with_sentiment_risk_train.csv"

assert os.path.isdir(WORKDIR), f"Folder not found: {WORKDIR}  - upload the package to MyDrive as 'btc_rl_task'"
os.chdir(WORKDIR)

if not os.path.exists(CSV):
    assert os.path.exists(GZ), f"Dataset not found: {GZ}"
    t0 = time.time()
    with gzip.open(GZ, "rb") as fi, open(CSV, "wb") as fo:
        shutil.copyfileobj(fi, fo, length=8 * 1024 * 1024)
    print(f"Decompressed in {time.time() - t0:.0f}s")
else:
    print("Dataset already restored")

for d in ["data/1min/alpha101", "output/1min", "trained_agents"]:
    os.makedirs(d, exist_ok=True)

print(f"{CSV}  ({os.path.getsize(CSV) / 1e6:,.0f} MB)")
print("working directory:", os.getcwd())

## Sanity check

Confirms size, date range and required columns, and that every input the models read
actually varies. A column with one distinct value is a dead feature.

In [ ]:
import pandas as pd

df = pd.read_csv(CSV)
print(f"rows x cols : {df.shape[0]:,} x {df.shape[1]}")
print(f"date range  : {df.system_time.iloc[0]}  ->  {df.system_time.iloc[-1]}")

need = ["midpoint", "spread", "buys", "sells",
        "bids_distance_3", "asks_distance_3", "bids_notional_3", "asks_notional_3",
        "sentiment_score", "risk_score"]
missing = [c for c in need if c not in df.columns]
print("missing required columns:", missing or "none")

print("\ndistinct values per required column:")
for c in need:
    if c in df.columns:
        n = df[c].nunique()
        print(f"  {c:<18}{n:>11,}{'   <-- DEAD FEATURE' if n == 1 else ''}")

print("\nsentiment_score range:", round(df.sentiment_score.min(), 3), "-", round(df.sentiment_score.max(), 3))
print("risk_score      range:", round(df.risk_score.min(), 3), "-", round(df.risk_score.max(), 3))
print(f"dataframe memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
del df

## Optional — 5-minute fallback

**Skip this unless notebook 3 runs out of memory.** It aggregates the 1,278,720
one-minute bars into 255,744 five-minute bars (~5x less memory). Sentiment and risk
are daily signals, so they are unaffected; only price granularity coarsens.

Bars are rebuilt from each minute's high/low, the same way the 1-minute file was
built, so `best_bid`, `best_ask`, `spread` and `bids/asks_distance_3` stay consistent.

**If you use it:** in notebooks 2 and 3 enable the `config_5min()` / `process_5min()`
cells, and set the timeframe to `'5min'` in the driver cells of notebooks 4, 5 and 6.
Report which resolution your results use.

In [ ]:
RUN_5MIN_FALLBACK = False   # set True only if notebook 3 runs out of memory

if RUN_5MIN_FALLBACK:
    df = pd.read_csv(CSV, parse_dates=["system_time"])
    b = df.system_time.dt.floor("5min")

    hi = (df.midpoint + df.spread / 2).groupby(b).max()
    lo = (df.midpoint - df.spread / 2).groupby(b).min()
    mid = (hi + lo) / 2

    g = df.groupby(b)
    out = pd.DataFrame({"system_time": hi.index})
    out["midpoint"] = mid.values
    out["spread"] = (hi - lo).values
    for c in ["buys", "sells", "trades"]:
        out[c] = g[c].sum().values
    for side in ["bids", "asks"]:
        for i in range(5):
            out[f"{side}_distance_{i}"] = g[f"{side}_distance_{i}"].first().values
        for i in range(5):
            out[f"{side}_notional_{i}"] = g[f"{side}_notional_{i}"].mean().values
    # per-bar near-touch values, recomputed at 5-minute resolution
    out["bids_distance_3"] = ((lo - mid) / mid).values
    out["asks_distance_3"] = ((hi - mid) / mid).values
    out["best_bid"], out["best_ask"] = lo.values, hi.values
    # daily signals are constant within a day: take the value, never sum
    for c in ["sentiment_score", "risk_score", "sentiment_weighted", "risk_weighted", "tweet_volume"]:
        out[c] = g[c].last().values

    out = out[[c for c in df.columns if c in out.columns]]
    DST5 = f"{WORKDIR}/data/BTC_5min_with_sentiment_risk_train.csv"
    out.to_csv(DST5, index=False)
    print(f"{len(out):,} rows -> {DST5}")
    del df, out
else:
    print("Skipped (RUN_5MIN_FALLBACK = False)")

## Next: run notebooks 2 → 6 in order

Open each from the `btc_rl_task` Drive folder and use **Runtime → Run all**. Each one
mounts Drive and switches to `btc_rl_task/` in its first cell. They are
pre-configured for the 1-minute data.

| Notebook | Stage | Writes |
|---|---|---|
| `2_data_splitter` | Chronological 70 / 15 / 15 split | `data/1min/*_train_70.csv`, `_val_15`, `_test_15` |
| `3_alpha_signals_generator` | Alpha101 factors | `data/1min/alpha101/alpha101_{train,val,test}.npy` |
| `4_rnn_trainer` | Recurrent factor model → 8 trend factors | `output/1min/*_predictions.npy`, `best_model.pth` |
| `5_erl_trainer` | PPO agent | `trained_agents/.../1min/PPO/actor.pth`, `critic.pth` |
| `6_erl_evaluator` | Backtest and metrics | evaluation results and plots |

The split is chronological, so it falls as:

| Split | Period | Market regime |
|---|---|---|
| Train (70%) | 2024-01 → 2025-09 | halving bull run |
| Validation (15%) | 2025-09 → 2026-01 | all-time high and topping |
| **Test (15%)** | **2026-01 → 2026-06** | **crash and capitulation** |